# **ML APPROACH**

**LOADING AND CLEANING COMMODITY DATA**

In [117]:
# Libraries

from pathlib import Path
import pandas as pd
import numpy as np

In [118]:
print(Path.cwd())

c:\Users\mason\OneDrive\Desktop\AgGenius\wasde_analysis\notebooks\models


In [119]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'commodity_prod'

print(Path.cwd())
print(DATA_DIR.exists())

c:\Users\mason\OneDrive\Desktop\AgGenius\wasde_analysis\notebooks\models
True


In [120]:
commodity_files = {
    'Corn': 'corn_prod.csv',
    'Cotton': 'cott_prod.csv',
    'Hard Red Spring Wheat': 'hrs_prod.csv',
    'Hard Red Winter Wheat': 'hrw_prod.csv',
    'Oats': 'oats_prod.csv',
    'Rice (Rough)': 'rr_prod.csv',
    'Soybean Meal': 'soy_meal_prod.csv',
    'Soybean Oil': 'soy_oil_prod.csv',
    'Soybeans': 'soybean_prod (2).csv',
    'Sugar': 'sugar_prod.csv',
    'Wheat': 'wht_prod.csv',
}

raw_data = {}
for commodity, filename in commodity_files.items():
    path = DATA_DIR / filename
    raw_data[commodity] = pd.read_csv(path, index_col=0, parse_dates=False)
    print(commodity, '->', raw_data[commodity].shape)

Corn -> (567, 15)
Cotton -> (567, 15)
Hard Red Spring Wheat -> (346, 15)
Hard Red Winter Wheat -> (346, 15)
Oats -> (567, 15)
Rice (Rough) -> (567, 15)
Soybean Meal -> (567, 15)
Soybean Oil -> (567, 15)
Soybeans -> (567, 15)
Sugar -> (567, 15)
Wheat -> (567, 15)


In [121]:
for commodity, d in raw_data.items():
    if 'ReliabilityProjection' in d.columns:
        d.drop(columns='ReliabilityProjection', inplace=True)
    if d['ReportDate'].dtype == str:
        d['ReportDate'] = pd.to_datetime(d['ReportDate'])

# Confirm the cleanup landed everywhere
for commodity, d in raw_data.items():
    print(commodity, d.shape, d['ReportDate'].dtype)

Corn (567, 14) str
Cotton (567, 14) str
Hard Red Spring Wheat (346, 14) str
Hard Red Winter Wheat (346, 14) str
Oats (567, 14) str
Rice (Rough) (567, 14) str
Soybean Meal (567, 14) str
Soybean Oil (567, 14) str
Soybeans (567, 14) str
Sugar (567, 14) str
Wheat (567, 14) str


In [122]:
print(f"Loaded {len(raw_data)} of {len(commodity_files)} expected commodities")
missing = set(commodity_files.keys()) - set(raw_data.keys())
print('Missing:', missing if missing else 'none')

Loaded 11 of 11 expected commodities
Missing: none


In [123]:
excluded_commodities = ['Hard Red Spring Wheat', 'Hard Red Winter Wheat']

raw_data_modeling = {k: v for k, v in raw_data.items() if k not in excluded_commodities}

print(f"Modeling on {len(raw_data_modeling)} commodities: {list(raw_data_modeling.keys())}")

Modeling on 9 commodities: ['Corn', 'Cotton', 'Oats', 'Rice (Rough)', 'Soybean Meal', 'Soybean Oil', 'Soybeans', 'Sugar', 'Wheat']


DateTime conversion

In [124]:
for commodity, d in raw_data_modeling.items():
    if not pd.api.types.is_datetime64_any_dtype(d['ReportDate']):
        d['ReportDate'] = pd.to_datetime(d['ReportDate'])
    print(commodity, d['ReportDate'].dtype)

Corn datetime64[us]
Cotton datetime64[us]
Oats datetime64[us]
Rice (Rough) datetime64[us]
Soybean Meal datetime64[us]
Soybean Oil datetime64[us]
Soybeans datetime64[us]
Sugar datetime64[us]
Wheat datetime64[us]


C:\Users\mason\AppData\Local\Temp\ipykernel_16172\2340286167.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d['ReportDate'] = pd.to_datetime(d['ReportDate'])
C:\Users\mason\AppData\Local\Temp\ipykernel_16172\2340286167.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d['ReportDate'] = pd.to_datetime(d['ReportDate'])
C:\Users\mason\AppData\Local\Temp\ipykernel_16172\2340286167.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d['ReportDate'] = pd.to_datetime(d['ReportDate'])
C:\Users\mason\AppData\Local\Temp\ipykernel_16172\2340286167.py:3: UserWarning: Could not i

**Excluded:** Hard Red Spring Wheat, Hard Red Winter Wheat

WASDE never publishes a finalized (ProjEstFlag = NaN) production figure at the wheat-subclass level — confirmed across both raw commodity files and the master concatenated source, under every label variant. 

Only aggregate 'Wheat' production is ever finalized. Without ground truth, these two can't be validated as standalone nowcasting targets. 

**Possible future extension:** use HRS/HRW live estimates as input features for predicting total Wheat's finalized value, rather than as targets themselves

# **Run 'build_vintages' function across all commodities**

In [125]:
def build_vintages(df, commodity_name=None):
    """
    Given a cleaned WASDE dataframe for one commodity, build the vintage
    table: every live (Proj./Est.) estimate tagged with its horizon to
    finalization and the eventual final value.
    """
    df = df.copy()

    # First-finalized appearance per market year
    finals_info = (
        df[df['ProjEstFlag'].isna()]
          .sort_values('ReportDate')
          .groupby('MarketYear')
          .head(1)
          .set_index('MarketYear')[['ReportDate', 'Value']]
          .rename(columns={'ReportDate': 'final_date', 'Value': 'final_value'})
    )

    vintages = df[df['ProjEstFlag'].notna()].copy()
    vintages['final_value'] = vintages['MarketYear'].map(finals_info['final_value'])
    vintages['final_date'] = vintages['MarketYear'].map(finals_info['final_date'])

    vintages['horizon_months'] = (
        (vintages['final_date'].dt.year - vintages['ReportDate'].dt.year) * 12
        + (vintages['final_date'].dt.month - vintages['ReportDate'].dt.month)
    )

    # Drop any market year without a confirmed final value (still-live years)
    vintages = vintages.dropna(subset=['final_value', 'final_date']).copy()

    if commodity_name:
        vintages['Commodity'] = commodity_name

    return vintages, finals_info

In [126]:
all_vintages = []
finals_by_commodity = {}

for commodity, d in raw_data_modeling.items():
    v, finals_info = build_vintages(d, commodity_name=commodity)
    all_vintages.append(v)
    finals_by_commodity[commodity] = finals_info
    print(f"{commodity}: {len(v)} vintage rows, {v['MarketYear'].nunique()} finalized market years, max horizon {v['horizon_months'].max()}")

master_vintages = pd.concat(all_vintages, ignore_index=True)
print('\nTotal pooled rows:', master_vintages.shape)

Corn: 344 vintage rows, 16 finalized market years, max horizon 24.0
Cotton: 322 vintage rows, 16 finalized market years, max horizon 24.0
Oats: 344 vintage rows, 16 finalized market years, max horizon 24.0
Rice (Rough): 344 vintage rows, 16 finalized market years, max horizon 24.0
Soybean Meal: 344 vintage rows, 16 finalized market years, max horizon 24.0
Soybean Oil: 344 vintage rows, 16 finalized market years, max horizon 24.0
Soybeans: 344 vintage rows, 16 finalized market years, max horizon 24.0
Sugar: 344 vintage rows, 16 finalized market years, max horizon 24.0
Wheat: 344 vintage rows, 16 finalized market years, max horizon 24.0

Total pooled rows: (3074, 17)


In [127]:
for commodity, d in raw_data_modeling.items():
    n_null_flag = d['ProjEstFlag'].isna().sum()
    unique_flags = d['ProjEstFlag'].unique()
    print(f"{commodity}: {n_null_flag} finalized rows | flags: {list(unique_flags)}")

Corn: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Cotton: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Oats: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Rice (Rough): 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Soybean Meal: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Soybean Oil: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Soybeans: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Sugar: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']
Wheat: 189 finalized rows | flags: [nan, 'Est.', 'Proj.']


# **ESTABLISHING A BASELINE**

# **FEATURE ENGINEERING**

# **TRAIN TEST SPLIT**

# **FIT POOLED GRADIENT BOOSTING MODEL**

LightGBM or XGBoost on the full 3,074-row pooled set. Start simple (few features, shallow trees) before adding complexity.

# **EVALUATE AGAINST BASELINE**

Compare LOO MAE, pooled ML vs. raw WASDE estimate

# **RESULTS WRITE UP**